# HDB Resale Flat Price ETL Pipeline

This notebook is the execution entry point for the HDB Senior Data Engineer technical test. It runs the ETL pipeline end-to-end and produces the mandatory raw, cleaned, transformed, failed and hashed outputs.

The implementation avoids manual file edits and keeps the logic modular under the `src/` package.


## 1. Import pipeline modules

The notebook uses project modules so that the ETL logic is maintainable and not trapped in notebook-only cells.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.utils import load_config, ensure_dirs, project_root, write_csv
from src.extract import load_raw_files
from src.profile import profile_dataset
from src.validate import validate_dataset, handle_duplicates, flag_price_anomalies
from src.transform import transform_dataset
from src.hash_identifier import add_hashed_identifier

print(f'Project root: {PROJECT_ROOT}')


## 2. Load configuration

The date range, output directories, current date for remaining lease calculation and anomaly settings are parameterised in `config/pipeline_config.yaml`.


In [ ]:
config = load_config()
ensure_dirs(config)
config


## 3. Extract and combine raw datasets

The pipeline downloads HDB resale flat price data from data.gov.sg Collection 189 when raw files are not already present. Raw source files are stored under `data/raw/` without manual modifications.

The master dataset is filtered to the required assignment period: `2012-01` to `2016-12`.


In [ ]:
master_df = load_raw_files(config)
print('Master records:', len(master_df))
print('Columns:', list(master_df.columns))
master_df.head()


## 4. Data profiling

This creates a column-level profile including record count, null count, null percentage, distinct count and sample values.


In [ ]:
profile_df = profile_dataset(master_df, config)
profile_df


## 5. Data validation

Validation covers date, town, flat type, flat model, storey range, resale price, floor area, lease commencement date, block and street name. Failed records are retained with `failure_reason`.


In [ ]:
cleaned_df, failed_df, validation_summary = validate_dataset(master_df, config)
print('Cleaned before duplicate handling:', len(cleaned_df))
print('Failed before duplicate handling:', len(failed_df))
validation_summary


## 6. Composite-key duplicate handling

The composite key uses all columns except `resale_price`. If two records share the same composite key, the pipeline keeps the higher resale price and moves the lower-priced duplicate to the failed dataset.


In [ ]:
cleaned_df, failed_df = handle_duplicates(cleaned_df, failed_df, config)
print('Cleaned after duplicate handling:', len(cleaned_df))
print('Failed after duplicate handling:', len(failed_df))


## 7. Potential resale price anomaly flagging

The heuristic uses IQR within `month`, `town` and `flat_type`. Records outside the IQR thresholds are flagged as potential anomalies for review, not automatically removed.


In [ ]:
cleaned_df, anomaly_summary = flag_price_anomalies(cleaned_df, config)
anomaly_summary


## 8. Transformation

The transformation step recomputes remaining lease and creates the required resale identifier:

`S + 3 block digits + 2 average-price digits + 2 month digits + first town character`


In [ ]:
transformed_df = transform_dataset(cleaned_df, config)
transformed_df[['month', 'town', 'flat_type', 'block', 'resale_price', 'remaining_lease_recomputed', 'resale_identifier']].head()


## 9. Hash resale identifier

The resale identifier is hashed using SHA-256, an irreversible deterministic hashing algorithm.


In [ ]:
hashed_df = add_hashed_identifier(transformed_df)
hashed_df[['resale_identifier', 'hashed_resale_identifier']].head()


## 10. Export mandatory outputs

The notebook writes cleaned, transformed, failed and hashed datasets to their required output folders.


In [ ]:
write_csv(cleaned_df, project_root() / config['cleaned_dir'] / 'hdb_resale_cleaned.csv')
write_csv(transformed_df, project_root() / config['transformed_dir'] / 'hdb_resale_transformed.csv')
write_csv(failed_df, project_root() / config['failed_dir'] / 'hdb_resale_failed.csv')
write_csv(hashed_df, project_root() / config['hashed_dir'] / 'hdb_resale_hashed.csv')

summary = {
    'master_records': len(master_df),
    'cleaned_records': len(cleaned_df),
    'failed_records': len(failed_df),
    'transformed_records': len(transformed_df),
    'hashed_records': len(hashed_df),
    'potential_price_anomalies': int(cleaned_df['potential_price_anomaly'].sum())
}
summary


## 11. Output location check


In [ ]:
for folder in ['raw_dir', 'cleaned_dir', 'transformed_dir', 'failed_dir', 'hashed_dir', 'outputs_dir']:
    path = project_root() / config[folder]
    print(f'\n{folder}: {path}')
    for item in sorted(path.glob('*')):
        print(' -', item.name)
